# EMAS — Training & Inference Notebook (Colab)

This notebook provides a reproducible Colab pipeline to run our EMAS project end-to-end:
1. Install dependencies
2. Mount Google Drive (datasets + pretrained weights)
3. (Recommended) copy Cityscapes ZIPs to local SSD for faster I/O
4. Unzip Cityscapes locally
5. Run training (`train.py`) or the hard-negative variant (`train_HN.py`)

> Tip: keep the code cells **exactly as provided** below and only edit the paths (Drive location, ckpt output folder).

## 1) Install dependencies

We install all Python packages required by EMAS:
- training framework: `lightning`
- core DL stack: `torch`, `torchvision`
- metrics / visualization: `scikit-learn`, `matplotlib`, `visdom`
- OOD metrics utilities: `ood-metrics` (if you use the standard library version)
- transformer / LoRA utilities: `timm`, `peft`, `transformers`

If you get version conflicts, restart the runtime and re-run this cell.We used cityscapes dataset directly on teh local memory of colab, we download the zip from the drive and then unzip it.

In [ ]:
!pip install lightning
!pip install torch torchvision
!pip install "opencv-python<4.10"
!pip install scikit-learn matplotlib visdom pillow
!pip install ood-metrics
!pip install -q timm peft lightning transformers

## 2) Mount Google Drive

We store large assets on Drive:
- Cityscapes ZIPs
- CNP train/val ZIPs
- pretrained EoMT weights (`.bin`)
- (optional) checkpoints output

Mount Drive to make these files visible from Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3) Recommended: Copy Cityscapes ZIPs to local SSD (“ZIP mode”)

Colab local SSD is significantly faster than reading large ZIPs directly from Drive.

This setup cell:
- ensures Drive is mounted
- defines Drive paths and local paths
- copies Cityscapes ZIPs from Drive → local SSD (without unzipping yet)

At the end, you should see:
- `PATH_CITY = '/content/Cityscapes_Local'`

> If the ZIPs are already present locally, the copy step will be skipped automatically.

In [ ]:
# @title ZIP copy on SSD Local
import os
import shutil
import time
from google.colab import drive

# --- PERCORSI DRIVE (I tuoi originali) ---
DRIVE_CITY_DIR = "/content/drive/MyDrive/Anomaly_Segmentation/Cityscapes"

# --- PERCORSI LOCALI ---
LOCAL_CITY_DIR = "/content/Cityscapes_Local"

def copy_zip_to_local(zip_name, drive_dir, local_dest):
    source = os.path.join(drive_dir, zip_name)
    dest = os.path.join(local_dest, zip_name)

    if not os.path.exists(source):
        print(f"⚠️ ERRORE: Non trovo {source} su Drive!")
        return

    if os.path.exists(dest):
        print(f"✅ {zip_name} già presente in locale.")
        return

    print(f"⏳ Copia di {zip_name} in corso... (da Drive a SSD Locale)")
    start = time.time()
    shutil.copy(source, dest)
    print(f"✅ Copiato in {time.time()-start:.1f}s")

os.makedirs(LOCAL_CITY_DIR, exist_ok=True)

print("\n--> Setup Cityscapes...")
copy_zip_to_local("leftImg8bit_trainvaltest.zip", DRIVE_CITY_DIR, LOCAL_CITY_DIR)
copy_zip_to_local("gtFine_trainvaltest.zip", DRIVE_CITY_DIR, LOCAL_CITY_DIR)


print("\n" + "="*50)
print("SETUP COMPLETED!")
print(f"PATH_CITY = '{LOCAL_CITY_DIR}'")
print("="*50)

## 4) Unzip Cityscapes locally

Now we unzip the Cityscapes archives into the local folder:
- `leftImg8bit_trainvaltest.zip` (images)
- `gtFine_trainvaltest.zip` (annotations)

This produces the standard Cityscapes directory structure under:
`/content/Cityscapes_Local/leftImg8bit/...` and `/content/Cityscapes_Local/gtFine/...`

Finally, we list the local directory to confirm everything is in place.

Remember to always write yes/y when asked

In [ ]:
# Path of zipped Cityscapes
CITY_ROOT = "/content/Cityscapes_Local"

# unzip immagies
!unzip -q /content/Cityscapes_Local/leftImg8bit_trainvaltest.zip -d $CITY_ROOT

# unzip annotations
!unzip -q /content/Cityscapes_Local/gtFine_trainvaltest.zip -d $CITY_ROOT

# check sanity
!ls $CITY_ROOT

## 5) Train the base EMAS model (`train.py`)

This cell launches the standard training pipeline.

Key arguments (high-level):
- `--city_root`: local Cityscapes path (fast I/O)
- `--cnp_zip_train`, `--cnp_zip_val`: CNP cut-and-paste ZIPs (Drive)
- `--init_from_eomt_bin`: pretrained EoMT weights (Drive)
- `--mix_city`, `--mix_cnp`: IID/OE sampling ratio inside each epoch
- `--m_in`, `--T`, `--warmup_epochs`: energy / calibration-related hyperparameters

**Important:** set `--ckpt_dir` to a folder where you want checkpoints to be saved.

In [ ]:
!python train.py \
  --city_root "/content/Cityscapes_Local" \
  --cnp_zip_train "/content/drive/MyDrive/Anomaly_Segmentation/cnp_train.zip" \
  --cnp_zip_val   "/content/drive/MyDrive/Anomaly_Segmentation/cnp_val.zip" \
  --init_from_eomt_bin "/content/drive/MyDrive/Anomaly_Segmentation/MaskArchitectureAnomaly_CourseProject/trained_models/eomt_cityscapes.bin" \
  --batch_city 1 --batch_cnp 1 \
  --mix_city 3 --mix_cnp 1 \
  --steps_per_epoch 400 --max_epochs 35 --val_limit 500 \
  --lr 1e-5 \
  --m_in -12.0 --T 1.0 \
  --warmup_epochs 2 \
  #modify ckpt_dir adding the path in which you want to safe the checkpoints 
  #--ckpt_dir ""

## 6) Train the hard-negative variant (`train_HN.py`)

This cell runs the Hard-Negative (HN) version of training.

Use this when you want stronger pressure on “difficult” negatives / anomalies.
The arguments mirror the base training script, so you can keep the same setup and only swap the entrypoint.

**Important:** as above, set `--ckpt_dir` to your desired checkpoint output folder.

In [ ]:
!python train_HN.py \
  --city_root "/content/Cityscapes_Local" \
  --cnp_zip_train "/content/drive/MyDrive/Anomaly_Segmentation/cnp_train.zip" \
  --cnp_zip_val   "/content/drive/MyDrive/Anomaly_Segmentation/cnp_val.zip" \
  --init_from_eomt_bin "/content/drive/MyDrive/Anomaly_Segmentation/MaskArchitectureAnomaly_CourseProject/trained_models/eomt_cityscapes.bin" \
  --batch_city 1 --batch_cnp 1 \
  --mix_city 3 --mix_cnp 1 \
  --steps_per_epoch 400 --max_epochs 35 --val_limit 500 \
  --lr 1e-5 \
  --m_in -12.0 --T 1.0 \
  --warmup_epochs 2 \
  #modify ckpt_dir adding the path in which you want to safe the checkpoints 
  #--ckpt_dir ""

## Next steps (after training)

After training completes, you should:
1. Locate the best checkpoint in your `--ckpt_dir`
2. Run evaluation / temperature calibration following the steps in eval.ipynb

This notebook focuses on the **Colab setup + training execution**. For evaluation/inference, refer to the corresponding scripts in the repository.